# 第2课：把July PDSI加入Iowa玉米产量模型

这是课程作业建模过程的第二步。本课会把7月已经观察到的干旱/湿润信息加入产量模型，并与第1课的单纯趋势模型比较。

本课仍然不会加入期货价格、套保策略或Monte Carlo。请从上到下逐个运行代码单元格。

## 0. 本课问题

第1课的模型只有时间趋势：

$$Yield_t = \beta_0 + \beta_1 TrendIndex_t + \varepsilon_t$$

第2课增加July PDSI：

$$Yield_t = \beta_0 + \beta_1 TrendIndex_t + \beta_2 JulyPDSI_t + \varepsilon_t$$

我们要回答：在控制长期技术趋势后，7月干旱/湿润信息是否能改善产量预测？

## 1. 导入工具

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

START_YEAR = 1996
FORECAST_YEAR = 2026

print('工具导入成功。')

## 2. 建立产量与天气数据表

- Iowa corn yield来自USDA NASS；
- July PDSI来自NOAA NCEI；
- PDSI通常为负表示偏干，为正表示偏湿。

为了让Notebook可以单独运行，本教学文件内置了已经核对的30年数据。最终项目仍保存完整CSV和来源URL。

In [ ]:
years = list(range(1996, 2026))
yield_bu_per_acre = [
    138.0, 138.0, 145.0, 149.0, 144.0, 146.0, 163.0, 157.0,
    181.0, 173.0, 166.0, 171.0, 171.0, 181.0, 165.0, 172.0,
    137.0, 164.0, 178.0, 192.0, 203.0, 202.0, 196.0, 198.0,
    177.0, 204.0, 200.0, 201.0, 211.0, 210.0
]
july_pdsi = [
    1.51, -0.36, 2.39, 3.10, 1.11, -0.37, 0.06, 0.78,
    1.47, -0.21, -2.39, 1.27, 6.45, 3.98, 6.69, -0.23,
    -3.56, -0.68, 2.27, 3.18, 3.50, 2.06, 1.53, 4.70,
    -0.36, -1.43, -0.89, -1.99, 2.43, 2.54
]

data = pd.DataFrame({
    'year': years,
    'yield_bu_per_acre': yield_bu_per_acre,
    'july_pdsi': july_pdsi,
})
data.head()

## 3. 检查数据并建立Trend Index

每个crop year必须同时拥有一个产量和一个July PDSI。

In [ ]:
assert len(data) == 30, '必须有30个历史年份。'
assert data['year'].tolist() == list(range(1996, 2026)), '年份必须为1996–2025。'
assert not data.isna().any().any(), '数据不能有缺失值。'
assert not data['year'].duplicated().any(), '年份不能重复。'

data['trend_index'] = data['year'] - START_YEAR

print('检查通过：30个年份，产量与PDSI一一对应。')
data[['year', 'yield_bu_per_acre', 'july_pdsi', 'trend_index']].tail()

## 4. 建立多元回归矩阵

每一行的解释变量是：

```text
[1, Trend Index, July PDSI]
```

第一列的1用于估计截距。OLS在矩阵形式下为：

$$\hat{\beta}=(X'X)^{-1}X'y$$

代码使用 `numpy.linalg.lstsq`，因为它在数值上比直接计算矩阵逆更稳定。

In [ ]:
X_pdsi = np.column_stack([
    np.ones(len(data)),
    data['trend_index'].to_numpy(dtype=float),
    data['july_pdsi'].to_numpy(dtype=float),
])
y = data['yield_bu_per_acre'].to_numpy(dtype=float)

print('X矩阵形状:', X_pdsi.shape)
print('前3行：')
print(X_pdsi[:3])

## 5. 估计Trend + July PDSI模型

In [ ]:
beta_pdsi, _, _, _ = np.linalg.lstsq(X_pdsi, y, rcond=None)
intercept_pdsi, trend_beta_pdsi, pdsi_beta = beta_pdsi

print(f'Intercept = {intercept_pdsi:.4f}')
print(f'Trend coefficient = {trend_beta_pdsi:.4f}')
print(f'July PDSI coefficient = {pdsi_beta:.4f}')
print()
print(
    f'Yield = {intercept_pdsi:.4f} '    f'+ {trend_beta_pdsi:.4f} × Trend Index '    f'+ {pdsi_beta:.4f} × July PDSI + Residual'
)

### 怎样解释PDSI系数？

如果PDSI系数约为1.91，表示：

> 在控制长期技术趋势以后，July PDSI每提高1点，历史样本中的Iowa玉米产量平均高约1.91 bu/acre。

这是历史统计关系，不应写成PDSI必然造成产量变化的因果结论。

## 6. 计算天气模型的拟合产量和Residual

$$YieldResidual_t=ActualYield_t-PredictedYield_t$$

这些Residual之后会成为Monte Carlo的历史随机误差库，但本课暂时不抽样。

In [ ]:
data['pdsi_model_fitted_yield'] = X_pdsi @ beta_pdsi
data['pdsi_model_yield_residual'] = (
    data['yield_bu_per_acre'] - data['pdsi_model_fitted_yield']
)

data[[
    'year', 'yield_bu_per_acre', 'july_pdsi',
    'pdsi_model_fitted_yield', 'pdsi_model_yield_residual'
]].tail().round(2)

## 7. 建立统一的模型评估函数

为了公平比较模型，我们对每个模型计算：

- R-squared；
- Adjusted R-squared；
- In-sample RMSE。

In [ ]:
def fit_and_measure(frame, predictors):
    X = np.column_stack([
        np.ones(len(frame)),
        *[frame[column].to_numpy(dtype=float) for column in predictors],
    ])
    y_local = frame['yield_bu_per_acre'].to_numpy(dtype=float)
    beta, _, _, _ = np.linalg.lstsq(X, y_local, rcond=None)
    fitted = X @ beta
    residuals = y_local - fitted

    n = len(frame)
    p = X.shape[1]
    sse = float((residuals ** 2).sum())
    sst = float(((y_local - y_local.mean()) ** 2).sum())
    r_squared = 1 - sse / sst
    adjusted_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - p)
    rmse = float(np.sqrt(np.mean(residuals ** 2)))

    return {
        'predictors': predictors,
        'coefficients': beta,
        'fitted': fitted,
        'residuals': residuals,
        'r_squared': r_squared,
        'adjusted_r_squared': adjusted_r_squared,
        'rmse': rmse,
    }

trend_only_model = fit_and_measure(data, ['trend_index'])
trend_pdsi_model = fit_and_measure(data, ['trend_index', 'july_pdsi'])
print('模型估计完成。')

## 8. 用LOOCV检查样本外预测

只有30年数据，不能只看样本内R-squared。Leave-One-Out Cross-Validation会：

1. 暂时拿掉1年；
2. 用剩余29年估计模型；
3. 预测被拿掉的那1年；
4. 对30年全部重复。

LOOCV RMSE越低，样本外预测表现越好。

In [ ]:
def loocv_rmse(frame, predictors):
    errors = []
    for test_index in range(len(frame)):
        train_mask = np.arange(len(frame)) != test_index
        train = frame.loc[train_mask]

        X_train = np.column_stack([
            np.ones(len(train)),
            *[train[column].to_numpy(dtype=float) for column in predictors],
        ])
        y_train = train['yield_bu_per_acre'].to_numpy(dtype=float)
        beta, _, _, _ = np.linalg.lstsq(X_train, y_train, rcond=None)

        test_row = frame.iloc[test_index]
        x_test = np.array([1.0, *[float(test_row[column]) for column in predictors]])
        prediction = float(x_test @ beta)
        actual = float(test_row['yield_bu_per_acre'])
        errors.append(actual - prediction)

    return float(np.sqrt(np.mean(np.square(errors))))

trend_only_loocv = loocv_rmse(data, ['trend_index'])
trend_pdsi_loocv = loocv_rmse(data, ['trend_index', 'july_pdsi'])

print(f'Trend-only LOOCV RMSE = {trend_only_loocv:.4f}')
print(f'Trend + PDSI LOOCV RMSE = {trend_pdsi_loocv:.4f}')

## 9. 建立模型比较表

不能只因为加入PDSI后R-squared上升就选择它，因为增加任何变量通常都会提高R-squared。我们重点看Adjusted R-squared和LOOCV RMSE。

In [ ]:
model_comparison = pd.DataFrame([
    {
        'model': 'Trend only',
        'predictors': 'Trend Index',
        'r_squared': trend_only_model['r_squared'],
        'adjusted_r_squared': trend_only_model['adjusted_r_squared'],
        'in_sample_rmse': trend_only_model['rmse'],
        'loocv_rmse': trend_only_loocv,
    },
    {
        'model': 'Trend + July PDSI',
        'predictors': 'Trend Index + July PDSI',
        'r_squared': trend_pdsi_model['r_squared'],
        'adjusted_r_squared': trend_pdsi_model['adjusted_r_squared'],
        'in_sample_rmse': trend_pdsi_model['rmse'],
        'loocv_rmse': trend_pdsi_loocv,
    },
])

model_comparison.round(4)

### 当前结论

Trend + PDSI模型的LOOCV RMSE略低于Trend-only，因此PDSI提供了一点额外的样本外预测信息。改善幅度不大，所以报告中应写成“modest improvement”，不能夸大天气模型的准确性。

最终作业还会把PDSI与其他简约候选模型统一比较，但本课先完成PDSI这一步。

## 10. 展示2026年不同July PDSI下的产量预测

2026年Trend Index为30。7月之前我们不知道2026年的PDSI，因此不能把某一个PDSI情景当成确定预测。这里仅展示不同天气信号如何改变7月产量预测。

In [ ]:
pdsi_low_tercile = float(data['july_pdsi'].quantile(1 / 3))
pdsi_high_tercile = float(data['july_pdsi'].quantile(2 / 3))
pdsi_historical_mean = float(data['july_pdsi'].mean())
forecast_trend_index = FORECAST_YEAR - START_YEAR

weather_scenarios = pd.DataFrame({
    'scenario': [
        'Lower tercile boundary',
        'Neutral PDSI = 0',
        'Historical mean PDSI',
        'Upper tercile boundary',
    ],
    'july_pdsi': [
        pdsi_low_tercile,
        0.0,
        pdsi_historical_mean,
        pdsi_high_tercile,
    ],
})

weather_scenarios['forecast_2026_yield_bu_per_acre'] = (
    intercept_pdsi
    + trend_beta_pdsi * forecast_trend_index
    + pdsi_beta * weather_scenarios['july_pdsi']
)

weather_scenarios.round(2)

## 11. 保存本课作业结果

本课会保存历史拟合表、模型比较表、天气情景表和模型参数。

In [ ]:
output_dir = Path.cwd() / 'lesson_output' / 'step_02'
output_dir.mkdir(parents=True, exist_ok=True)

data.to_csv(output_dir / 'yield_pdsi_model_1996_2025.csv', index=False)
model_comparison.to_csv(output_dir / 'yield_model_comparison.csv', index=False)
weather_scenarios.to_csv(output_dir / '2026_pdsi_yield_scenarios.csv', index=False)

summary = {
    'selected_for_this_lesson': 'Trend + July PDSI',
    'equation': (
        f'Yield = {intercept_pdsi:.6f} + {trend_beta_pdsi:.6f} * TrendIndex '
        f'+ {pdsi_beta:.6f} * JulyPDSI + Residual'
    ),
    'intercept': float(intercept_pdsi),
    'trend_coefficient': float(trend_beta_pdsi),
    'july_pdsi_coefficient': float(pdsi_beta),
    'adjusted_r_squared': float(trend_pdsi_model['adjusted_r_squared']),
    'loocv_rmse_bu_per_acre': float(trend_pdsi_loocv),
    'interpretation_limit': 'Historical association, not a causal estimate.',
}

with (output_dir / 'step_02_summary.json').open('w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2)

print('第2课结果已保存到：', output_dir)

## 本课完成标准

运行正确时应该得到：

- Intercept约为 **138.3604**；
- Trend coefficient约为 **2.3187**；
- July PDSI coefficient约为 **1.9079**；
- Adjusted R-squared约为 **0.7793**；
- LOOCV RMSE约为 **11.9201 bu/acre**。

到这里停止。下一步才会详细整理Residual如何进入Monte Carlo随机抽样。